In [98]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score



In [ ]:
# Read the data
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.columns = df.columns.str.lower().str.replace(' ', '_')
category = list(df.dtypes[df.dtypes == 'object'].index)

for c in category:
    df[c] = df[c].str.lower().str.replace(' ', '_')

df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')
df.totalcharges = df.totalcharges.fillna(0)



In [ ]:
# Convert churn to binary
df.churn = (df.churn == 'yes').astype(int)

# Split the data
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

numerical = ['tenure', 'monthlycharges', 'seniorcitizen']
category = ['gender', 'partner', 'dependents',
            'phoneservice', 'multiplelines', 'internetservice',
            'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
            'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling',
            'paymentmethod']

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.churn.values
y_val = df_val.churn.values
y_test = df_test.churn.values
del df_train['churn']
del df_val['churn']
del df_test['churn']



In [ ]:
df_full_train = df_full_train.reset_index(drop=True)
df_full_train.churn.value_counts(normalize=True)

train_dicts = df_train[category + numerical].to_dict(orient='records')
val_dicts = df_val[category + numerical].to_dict(orient='records')
test_dicts = df_test[category + numerical].to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)
X_test = dv.transform(test_dicts)



In [130]:
# Define KFold
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=1)

# Initialize list to store roc_auc_scores
roc_auc_scores = []

# Train and evaluate the model using KFold
for train_idx, val_idx in kf.split(X_train):
    X_train_fold, X_val_fold = X_train[train_idx], X_train[val_idx]
    y_train_fold, y_val_fold = y_train[train_idx], y_train[val_idx]
    
    model = LogisticRegression()
    model.fit(X_train_fold, y_train_fold)
    
    y_pred_proba = model.predict_proba(X_val_fold)[:, 1]
    roc_auc = roc_auc_score(y_val_fold, y_pred_proba)
    roc_auc_scores.append(roc_auc)

# Compute average roc_auc_score
average_roc_auc = np.mean(roc_auc_scores)
print(f'Average ROC AUC Score: {average_roc_auc}')


/home/mohamed/mlops_project/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/mohamed/mlops_project/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/mod

Average ROC AUC Score: 0.8374442428555838


/home/mohamed/mlops_project/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [132]:
roc_auc_scores ,model,dv
C=1.0

In [133]:
def train(df_train,y_train,C=1.0):
    dicts=df_train[category +numerical].to_dict(orient='records')

    dv=DictVectorizer(sparse=False)
    X_train=df.fit_transform(dicts)

    model = LogisticRegression
    model.fit(X_train,y_train)

    return dv, model
def predict(df,dv,model):
    dicts = df[category+ numerical].to_dict(orient='records')

    X= dv.transform(dicts)
    y_pred=model.predict_proba(X)[:,1]

    return y_pred


In [134]:
import pickle


In [135]:
output_file= f'model_C={C}.bin'
output_file

'model_C=1.0.bin'

In [136]:
with open(output_file, 'wb') as f_out:
    pickle.dump((dv,model), f_out)


In [137]:
 df.columns

Index(['customerid', 'gender', 'seniorcitizen', 'partner', 'dependents',
       'tenure', 'phoneservice', 'multiplelines', 'internetservice',
       'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
       'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling',
       'paymentmethod', 'monthlycharges', 'totalcharges', 'churn'],
      dtype='object')